# EDA 03 — Calidad del input para los modelos 3D

**¿Qué estudia este notebook?**

Los tres métodos que vamos a comparar (NeXF, EndoGaussian, EndoDepthAndMotion) no consumen imágenes estáticas — consumen **secuencias de video con movimiento de cámara**. Para que funcionen necesitan:

1. **NeXF (NeRF):** múltiples vistas del mismo punto desde ángulos distintos. Necesita poses de cámara conocidas y suficiente solapamiento entre frames.
2. **EndoGaussian (Gaussian Splatting):** igual que NeRF pero además es muy sensible a la calidad de los frames individuales (blur, saturación).
3. **EndoDepthAndMotion (SLAM + Depth):** secuencia temporal continua con movimiento suave. Usa la consistencia temporal entre frames consecutivos para estimar profundidad.

Este notebook responde: **¿el video de SCARED es apto para estos tres métodos?** ¿Hay suficiente movimiento, suficiente solapamiento, los frames son nítidos?

---

### Estructura
1. Configuración e imports
2. Análisis del video `rgb.mp4` — resolución, duración, frames
3. Calidad individual de frames — blur, saturación, brillo
4. Movimiento entre frames — flujo óptico, baseline de cámara
5. Análisis de poses — suavidad de trayectoria, traslación acumulada
6. Solapamiento entre frames — qué fracción del GT es visible en cada frame
7. Tabla de aptitud por keyframe para cada modelo
8. Resumen ejecutivo

---
## Sección 1 — Configuración e imports

In [ ]:
import zipfile, io, json, tarfile
import numpy as np
import cv2
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from pathlib import Path
from scipy.spatial.transform import Rotation

# ── Cambia esta ruta si tus ZIPs están en otro lugar ──────────────────────────
SCARED_DIR = Path("E:/scared_raw")
# ─────────────────────────────────────────────────────────────────────────────

OUT_DIR  = Path("../outcomes/eda_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = SCARED_DIR / "dataset_1.zip"
DATASET  = "dataset_1"

# Keyframes que tienen carpeta data/ (keyframe_4 no la tiene por error de logging)
KF_WITH_VIDEO = ["keyframe_1", "keyframe_2", "keyframe_3", "keyframe_5"]

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"
print("ZIP encontrado:", ZIP_PATH)
print("Keyframes con video:", KF_WITH_VIDEO)

---
## Sección 2 — Análisis del video `rgb.mp4`

### ¿Qué es el `rgb.mp4`?

Después de capturar el keyframe con luz estructurada, el endoscopio se **mueve libremente** por el tejido mientras graba video normal (sin luz estructurada). Ese video es `rgb.mp4`.

El formato es especial: los frames de la cámara izquierda y derecha están **apilados verticalmente** — la mitad superior es la cámara izquierda, la mitad inferior es la derecha.

**Lo que necesitamos saber:**
- ¿Cuántos frames tiene? (define cuántos pares imagen-profundidad tenemos para entrenamiento)
- ¿Cuál es la resolución? (debe coincidir con 1280×1024 por cámara)
- ¿Cuánto dura en segundos? (un video muy corto puede no tener suficiente movimiento)
- ¿El framerate es consistente? (un framerate irregular complica SLAM)

In [ ]:
video_stats = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for kf in KF_WITH_VIDEO:
        video_path = f"{DATASET}/{kf}/data/rgb.mp4"

        # Extraer el video a un buffer en memoria y leerlo con OpenCV
        video_bytes = io.BytesIO(zf.read(video_path))

        # OpenCV necesita un archivo real — escribimos a un temp file mínimo
        import tempfile, os
        with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
            tmp.write(video_bytes.read())
            tmp_path = tmp.name

        cap = cv2.VideoCapture(tmp_path)
        fps        = cap.get(cv2.CAP_PROP_FPS)
        n_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height     = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration_s = n_frames / fps if fps > 0 else 0

        # Leer un frame de muestra (frame 0)
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame_sample = cap.read()
        cap.release()
        os.unlink(tmp_path)

        video_stats.append({
            'Keyframe':    kf.replace('keyframe_', 'KF'),
            'Frames':      n_frames,
            'FPS':         round(fps, 1),
            'Duracion_s':  round(duration_s, 1),
            'Width_px':    width,
            'Height_px':   height,
            'H_por_cam':   height // 2,   # altura de cada cámara individualmente
        })
        print(f"{kf}: {n_frames} frames @ {fps:.1f} fps  |  {width}x{height} px  |  {duration_s:.1f}s")

df_video = pd.DataFrame(video_stats)
print()
print(df_video.to_string(index=False))
df_video.to_csv(OUT_DIR / '22_video_metadata.csv', index=False)

### Cómo leer estos resultados

**`Frames`:** el número de pares imagen-profundidad disponibles para entrenamiento. Más frames = mejor, pero solo si hay suficiente movimiento entre ellos.

**`FPS`:** framerate. Un framerate alto (>25 fps) con movimiento de cámara lento significa que los frames consecutivos son casi idénticos — redundantes. Para NeRF/Gaussian Splatting conviene submuestrear (tomar 1 de cada N frames).

**`Height_px`:** el video tiene el doble de altura porque apila izquierda y derecha. `H_por_cam` es la altura real de cada imagen: debe ser 1024 para coincidir con el keyframe.

**`Duracion_s`:** una secuencia de ~6 segundos a 25 fps da ~150 frames — suficiente para NeRF si el movimiento cubre el espacio bien. Menos de 3 segundos puede ser insuficiente.

> **Para EndoDepthAndMotion:** necesita secuencias largas y continuas. Duraciones < 5s son un riesgo.
> **Para NeXF/EndoGaussian:** calidad sobre cantidad — 50 frames bien distribuidos pueden ser suficientes.

---
## Sección 3 — Calidad individual de frames

### ¿Por qué importa la calidad de cada frame?

Los modelos 3D asumen que los frames son representaciones fieles de la escena. Un frame borroso o sobrexpuesto contamina la reconstrucción. Las tres métricas que medimos son:

**Varianza del Laplaciano (nitidez):**
El operador Laplaciano detecta bordes. Si la imagen es nítida, los bordes son fuertes → la varianza del Laplaciano es alta. Si la imagen es borrosa (por movimiento rápido del endoscopio), los bordes son suaves → varianza baja.
- `> 100` → nítido
- `50–100` → aceptable
- `< 50` → borroso, problemático para los modelos

**Porcentaje de píxeles saturados:**
Píxeles con valor 255 en algún canal. Indica sobrexposición o especulares. Si es > 5% el frame tiene problemas de iluminación severos.

**Luminancia media:**
Un frame muy oscuro (< 50) o muy brillante (> 200) tendrá poco contraste útil.

Muestreamos **cada 10 frames** del video para no leer el archivo completo.

In [ ]:
SAMPLE_STEP = 10   # analizar 1 de cada 10 frames

fig, axes = plt.subplots(3, len(KF_WITH_VIDEO), figsize=(20, 10), sharex='col')

quality_summary = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for col, kf in enumerate(KF_WITH_VIDEO):
        video_path = f"{DATASET}/{kf}/data/rgb.mp4"
        video_bytes = io.BytesIO(zf.read(video_path))

        import tempfile, os
        with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
            tmp.write(video_bytes.read())
            tmp_path = tmp.name

        cap = cv2.VideoCapture(tmp_path)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        H_total  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        H_cam    = H_total // 2   # solo cámara izquierda (mitad superior)

        lap_vars, sat_pcts, lum_means = [], [], []
        frame_idxs = []

        for fi in range(0, n_frames, SAMPLE_STEP):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            if not ret:
                break

            left = frame[:H_cam]   # cámara izquierda
            gray = cv2.cvtColor(left, cv2.COLOR_BGR2GRAY)

            lap_var  = cv2.Laplacian(gray, cv2.CV_64F).var()
            sat_pct  = (gray > 250).mean() * 100
            lum_mean = gray.mean()

            lap_vars.append(lap_var)
            sat_pcts.append(sat_pct)
            lum_means.append(lum_mean)
            frame_idxs.append(fi)

        cap.release()
        os.unlink(tmp_path)

        lap_arr = np.array(lap_vars)
        sat_arr = np.array(sat_pcts)
        lum_arr = np.array(lum_means)

        quality_summary.append({
            'Keyframe':          kf.replace('keyframe_', 'KF'),
            'Frames_analizados': len(frame_idxs),
            'Lap_median':        round(float(np.median(lap_arr)), 1),
            'Lap_p10':           round(float(np.percentile(lap_arr, 10)), 1),
            'Sat_median_%':      round(float(np.median(sat_arr)), 2),
            'Lum_median':        round(float(np.median(lum_arr)), 1),
            'Frames_borrosos_%': round(float((lap_arr < 50).mean() * 100), 1),
        })

        kf_label = kf.replace('keyframe_', 'KF ')
        axes[0, col].plot(frame_idxs, lap_arr, color='steelblue', lw=1)
        axes[0, col].axhline(50,  color='red',    ls='--', lw=1, label='umbral blur')
        axes[0, col].axhline(100, color='orange', ls='--', lw=1, label='umbral ok')
        axes[0, col].set_title(kf_label, fontsize=11)
        axes[0, col].set_ylabel('Lap. Var.', fontsize=8)
        if col == 0: axes[0, col].legend(fontsize=7)

        axes[1, col].plot(frame_idxs, sat_arr, color='tomato', lw=1)
        axes[1, col].axhline(5, color='red', ls='--', lw=1, label='severo (5%)')
        axes[1, col].set_ylabel('Saturados (%)', fontsize=8)
        if col == 0: axes[1, col].legend(fontsize=7)

        axes[2, col].plot(frame_idxs, lum_arr, color='goldenrod', lw=1)
        axes[2, col].axhline(50,  color='blue',  ls='--', lw=1, label='muy oscuro')
        axes[2, col].axhline(200, color='red',   ls='--', lw=1, label='muy brillante')
        axes[2, col].set_ylabel('Luminancia', fontsize=8)
        axes[2, col].set_xlabel('Frame #', fontsize=8)
        if col == 0: axes[2, col].legend(fontsize=7)

plt.suptitle('Calidad de frames por keyframe (muestreo cada 10 frames)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / '23_calidad_frames.png', dpi=150, bbox_inches='tight')
plt.show()

df_quality = pd.DataFrame(quality_summary)
print(df_quality.to_string(index=False))
df_quality.to_csv(OUT_DIR / '23_calidad_frames.csv', index=False)

### Cómo leer estas gráficas

Cada columna es un keyframe. Hay tres filas:

**Fila 1 — Varianza del Laplaciano (nitidez):**
- El eje Y muestra la nitidez del frame. La línea roja es el umbral de blur severo (50), la naranja el umbral aceptable (100).
- Picos hacia abajo repentinos = el endoscopio hizo un movimiento rápido y el frame salió borroso.
- Si la mayoría de la curva está por debajo del umbral rojo → el keyframe tiene demasiado blur para entrenar bien.

**Fila 2 — Píxeles saturados:**
- Picos = momentos donde el tejido refleja mucho la luz (especulares intensos).
- Si los picos ocurren consistentemente en los mismos frames → el endoscopio está en una posición donde el ángulo de incidencia es especialmente problemático.

**Fila 3 — Luminancia media:**
- Debe mantenerse relativamente estable (entre 80–180). Caídas bruscas = el endoscopio se aleja del tejido o apunta a una zona muy oscura. Picos = acercamiento excesivo.
- La variabilidad de luminancia a lo largo del tiempo es otra medida de cuán difícil es el video para los modelos.

**`Frames_borrosos_%`** en la tabla: si es > 20%, el video tiene demasiados frames malos y se debe aplicar selección de frames antes de entrenar.

---
## Sección 4 — Movimiento entre frames (flujo óptico)

### ¿Por qué necesitamos analizar el movimiento?

Los modelos 3D necesitan movimiento de cámara para inferir la geometría: si el endoscopio no se mueve, todas las vistas son iguales y no hay información 3D.

Pero hay un balance:
- **Muy poco movimiento** → frames casi idénticos → el modelo no gana nueva información con cada frame.
- **Movimiento excesivo** → el frame sale borroso y la correspondencia entre frames se pierde.
- **Movimiento óptimo** → suficiente desplazamiento para ver el tejido desde ángulos distintos, pero sin blur.

### ¿Qué es el flujo óptico?

El **flujo óptico** (Optical Flow) mide cuánto se movió cada píxel entre dos frames consecutivos. Usamos el algoritmo de Farneback (densa) que calcula un vector de movimiento `(dx, dy)` para cada píxel.

Lo que reportamos es la **magnitud media del flujo**: cuántos píxeles se desplazaron en promedio entre un frame y el siguiente.

- `< 2 px/frame` → casi sin movimiento (redundante)
- `2–15 px/frame` → rango ideal para los modelos
- `> 20 px/frame` → movimiento excesivo, frames probablemente borrosos

In [ ]:
FLOW_STEP = 5   # calcular flujo entre frame i y frame i+FLOW_STEP (más eficiente)
SAMPLE_N  = 40  # solo los primeros 40 pares para que sea rápido

fig, axes = plt.subplots(1, len(KF_WITH_VIDEO), figsize=(20, 4), sharey=True)

flow_summary = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for ax, kf in zip(axes, KF_WITH_VIDEO):
        video_path = f"{DATASET}/{kf}/data/rgb.mp4"
        video_bytes = io.BytesIO(zf.read(video_path))

        import tempfile, os
        with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as tmp:
            tmp.write(video_bytes.read())
            tmp_path = tmp.name

        cap = cv2.VideoCapture(tmp_path)
        n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        H_total  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        H_cam    = H_total // 2

        flow_magnitudes = []
        prev_gray = None
        sampled = 0

        for fi in range(0, min(n_frames - FLOW_STEP, SAMPLE_N * FLOW_STEP), FLOW_STEP):
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, f1 = cap.read()
            if not ret: break
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi + FLOW_STEP)
            ret, f2 = cap.read()
            if not ret: break

            g1 = cv2.cvtColor(f1[:H_cam], cv2.COLOR_BGR2GRAY)
            g2 = cv2.cvtColor(f2[:H_cam], cv2.COLOR_BGR2GRAY)

            flow = cv2.calcOpticalFlowFarneback(
                g1, g2, None,
                pyr_scale=0.5, levels=3, winsize=15,
                iterations=3, poly_n=5, poly_sigma=1.2, flags=0
            )
            mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2).mean()
            flow_magnitudes.append(float(mag))
            sampled += 1

        cap.release()
        os.unlink(tmp_path)

        flow_arr = np.array(flow_magnitudes)
        flow_summary.append({
            'Keyframe':          kf.replace('keyframe_', 'KF'),
            'Flow_median_px':    round(float(np.median(flow_arr)), 2),
            'Flow_p90_px':       round(float(np.percentile(flow_arr, 90)), 2),
            'Frames_sin_mov_%':  round(float((flow_arr < 2).mean() * 100), 1),
            'Frames_excesivo_%': round(float((flow_arr > 20).mean() * 100), 1),
        })

        ax.plot(range(len(flow_arr)), flow_arr, color='darkcyan', lw=1.5)
        ax.axhline(2,  color='blue',   ls='--', lw=1, label='sin movimiento (<2px)')
        ax.axhline(15, color='orange', ls='--', lw=1, label='límite ok (15px)')
        ax.axhline(20, color='red',    ls='--', lw=1, label='excesivo (>20px)')
        ax.fill_between(range(len(flow_arr)), 2, 15,
                         where=[(2 <= v <= 15) for v in flow_arr],
                         alpha=0.15, color='green')
        ax.set_title(f"{kf.replace('keyframe_','KF ')}\nmediana={np.median(flow_arr):.1f} px",
                     fontsize=10)
        ax.set_xlabel('Par de frames (#)', fontsize=8)
        ax.set_ylabel('Flujo óptico (px)', fontsize=8)
        if ax == axes[0]: ax.legend(fontsize=7)

plt.suptitle('Magnitud del flujo óptico entre frames consecutivos (+5)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / '24_flujo_optico.png', dpi=150, bbox_inches='tight')
plt.show()

df_flow = pd.DataFrame(flow_summary)
print(df_flow.to_string(index=False))
df_flow.to_csv(OUT_DIR / '24_flujo_optico.csv', index=False)

### Cómo leer estas gráficas

El eje X es el número de par de frames analizados. El eje Y es cuántos píxeles se movieron en promedio.

La **banda verde** es la zona ideal (2–15 px de movimiento). Cuanto más tiempo pase la curva dentro de esa banda, mejor es el video para los modelos.

**Patrones que debes buscar:**
- **Curva plana baja** (< 2 px) → el endoscopio estuvo casi estático. Malo para NeRF: todas las vistas son casi iguales.
- **Picos altos** (> 20 px) → movimientos bruscos. En esos pares, los frames estarán borrosos. Si son frecuentes, hay que filtrarlos.
- **Curva que sube y baja suavemente** dentro de la banda verde → el video es ideal.

**`Frames_sin_mov_%`:** fracción de pares donde el movimiento es casi nulo. Si es > 30%, el video tiene mucha redundancia.

**`Frames_excesivo_%`:** fracción de pares con movimiento excesivo (potencialmente borrosos). Si es > 20%, hay que implementar selección de frames.

> **EndoDepthAndMotion** es el más sensible al movimiento excesivo porque usa flujo óptico internamente. Si hay picos > 20 px frecuentes, el SLAM puede perderse (tracking failure).

---
## Sección 5 — Análisis de poses de cámara (`frame_data.tar.gz`)

### ¿Qué información contienen las poses?

Cada frame del video tiene un archivo JSON con la **transformación rígida de la cámara**: una rotación (R) y una traslación (T) que describen cómo se movió el endoscopio desde el keyframe hasta ese frame.

Con estas poses podemos reconstruir la **trayectoria 3D** del endoscopio.

### ¿Por qué importa para los modelos?

- **NeXF y EndoGaussian** necesitan poses conocidas y precisas para cada frame. Si las poses son ruidosas o tienen errores grandes, la reconstrucción 3D será incorrecta.
- **EndoDepthAndMotion** estima sus propias poses durante el SLAM — pero las poses del dataset sirven como ground truth para validar.

### ¿Qué analizamos?

1. **Traslación acumulada:** ¿cuántos mm se movió el endoscopio en total? Una traslación mayor implica más variedad de vistas.
2. **Suavidad de la trayectoria:** ¿los saltos entre poses consecutivas son pequeños y consistentes? Un salto grande indica un error o un movimiento brusco.
3. **Distribución de rotaciones:** ¿el endoscopio solo traslada, o también rota? Las rotaciones grandes ayudan a observar el tejido desde ángulos distintos.

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, len(KF_WITH_VIDEO), hspace=0.4, wspace=0.3)

pose_summary = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for col, kf in enumerate(KF_WITH_VIDEO):
        tar_path = f"{DATASET}/{kf}/data/frame_data.tar.gz"
        tar_bytes = io.BytesIO(zf.read(tar_path))

        translations, rotations_deg = [], []

        with tarfile.open(fileobj=tar_bytes, mode='r:gz') as tar:
            members = sorted([m for m in tar.getmembers() if m.name.endswith('.json')],
                             key=lambda m: m.name)

            for m in members:
                data = json.load(tar.extractfile(m))

                # La pose puede estar en diferentes claves según el dataset
                # Buscamos R (rotación 3x3) y T (traslación 3x1)
                pose = data.get('camera-calibration', data)
                R_raw = pose.get('R', None)
                T_raw = pose.get('T', None)

                if R_raw is None or T_raw is None:
                    # Intentar con clave alternativa
                    R_raw = data.get('R', None)
                    T_raw = data.get('T', None)

                if R_raw is not None and T_raw is not None:
                    R_mat = np.array(R_raw).reshape(3, 3)
                    T_vec = np.array(T_raw).flatten()
                    translations.append(T_vec)

                    # Ángulo de rotación en grados
                    try:
                        angle = Rotation.from_matrix(R_mat).magnitude() * 180 / np.pi
                    except Exception:
                        angle = 0.0
                    rotations_deg.append(angle)

        if not translations:
            print(f"{kf}: no se encontraron poses R/T en los JSONs")
            continue

        trans_arr = np.array(translations)   # (N, 3)
        rot_arr   = np.array(rotations_deg)  # (N,)

        # Traslación acumulada (distancia entre poses consecutivas)
        deltas    = np.linalg.norm(np.diff(trans_arr, axis=0), axis=1)  # mm
        cum_dist  = np.concatenate([[0], np.cumsum(deltas)])

        pose_summary.append({
            'Keyframe':        kf.replace('keyframe_', 'KF'),
            'N_poses':         len(translations),
            'Dist_total_mm':   round(float(cum_dist[-1]), 1),
            'Delta_median_mm': round(float(np.median(deltas)), 3),
            'Delta_p90_mm':    round(float(np.percentile(deltas, 90)), 3),
            'Rot_median_deg':  round(float(np.median(rot_arr)), 2),
            'Saltos_grandes_%': round(float((deltas > deltas.mean() + 3*deltas.std()).mean() * 100), 1),
        })

        # Trayectoria 3D (proyección XY)
        ax_traj = fig.add_subplot(gs[0, col], projection='3d')
        ax_traj.plot(trans_arr[:, 0], trans_arr[:, 1], trans_arr[:, 2],
                     color='steelblue', lw=1.5)
        ax_traj.scatter(*trans_arr[0],  color='green', s=50, label='inicio')
        ax_traj.scatter(*trans_arr[-1], color='red',   s=50, label='fin')
        ax_traj.set_title(f"{kf.replace('keyframe_','KF ')}\n{len(translations)} poses",
                          fontsize=9)
        ax_traj.set_xlabel('X (mm)', fontsize=7)
        ax_traj.set_ylabel('Y (mm)', fontsize=7)
        ax_traj.set_zlabel('Z (mm)', fontsize=7)
        if col == 0: ax_traj.legend(fontsize=7)

        # Distancia entre poses consecutivas
        ax_delta = fig.add_subplot(gs[1, col])
        ax_delta.plot(deltas, color='darkorange', lw=1)
        ax_delta.axhline(deltas.mean(), color='steelblue', ls='--', lw=1,
                          label=f'media={deltas.mean():.2f}mm')
        ax_delta.axhline(deltas.mean() + 3*deltas.std(), color='red', ls=':', lw=1,
                          label='umbral salto grande')
        ax_delta.set_title(f"Δ traslación entre poses\ndist. total={cum_dist[-1]:.0f} mm",
                            fontsize=9)
        ax_delta.set_xlabel('Pose #', fontsize=8)
        ax_delta.set_ylabel('Δ (mm)', fontsize=8)
        if col == 0: ax_delta.legend(fontsize=7)
        ax_delta.grid(True, alpha=0.3)

plt.suptitle('Trayectorias y suavidad de poses — dataset_1', fontsize=13)
plt.savefig(OUT_DIR / '25_poses.png', dpi=150, bbox_inches='tight')
plt.show()

df_poses = pd.DataFrame(pose_summary)
print(df_poses.to_string(index=False))
df_poses.to_csv(OUT_DIR / '25_poses.csv', index=False)

### Cómo leer estos resultados

**Fila superior — trayectoria 3D:**
- El punto verde es donde empieza el movimiento, el rojo donde termina.
- Una trayectoria que cubre un área grande = más vistas distintas del tejido → mejor para NeRF y Gaussian Splatting.
- Una trayectoria muy compacta (el endoscopio casi no se movió) → el modelo no tendrá suficiente variedad de perspectivas.

**Fila inferior — Δ traslación entre poses consecutivas:**
- La línea azul es la media de desplazamiento por frame. Los puntos por encima de la línea roja punteada son saltos anómalos.
- **Un Δ muy pequeño** (< 0.1 mm/frame) con 25 fps significa que el endoscopio se movió solo ~2.5 mm por segundo — muy lento, muchos frames redundantes.
- **Saltos grandes** (> media + 3σ) pueden ser errores en las poses o movimientos bruscos reales. Si `Saltos_grandes_%` es > 5%, las poses tienen problemas.

**`Dist_total_mm`:** la distancia total recorrida por el endoscopio durante el video. Para NeXF se necesitan al menos ~20–30 mm de recorrido para tener suficiente parallax.

**`Rot_median_deg`:** la rotación promedio respecto al keyframe. Rotaciones > 15° pueden hacer que el mismo punto del tejido se vea muy distinto entre frames, dificultando la correspondencia.

---
## Sección 6 — Cobertura GT en los frames de video (`scene_points.tar.gz`)

### ¿Qué estamos midiendo aquí?

El keyframe tiene un depth map GT con 78% de cobertura. Pero los frames del video tienen el depth map **warpeado** desde el keyframe — no capturado de nuevo con structured light. Esto significa que:

1. Cuando el endoscopio se aleja del punto original, partes del tejido salen del campo de visión del keyframe y pierden GT.
2. El movimiento acumulado reduce la cobertura de GT conforme avanza el video.

**La pregunta clave:** ¿después de cuántos frames el GT del video es demasiado escaso para ser útil?

Leemos **solo los primeros 30 TIFFs** del `scene_points.tar.gz` (el archivo completo pesa ~2.6 GB) para caracterizar la tendencia.

### ¿Por qué importa para los modelos?

- **EndoDepthAndMotion** supervisa el depth con `scene_points.tiff`. Si la cobertura cae a < 30%, la supervisión es débil.
- **NeXF y EndoGaussian** usan los frames para reconstrucción, pero el GT de depth no es su supervisión directa. Sin embargo, la evaluación final sí usa el GT → conviene saber en qué frames el GT es confiable.

In [ ]:
N_TIFF_SAMPLE = 30   # leer solo los primeros N TIFFs (el tar completo pesa >2 GB)

fig, axes = plt.subplots(1, len(KF_WITH_VIDEO), figsize=(20, 4), sharey=True)

coverage_summary = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    for ax, kf in zip(axes, KF_WITH_VIDEO):
        sp_path  = f"{DATASET}/{kf}/data/scene_points.tar.gz"
        sp_bytes = io.BytesIO(zf.read(sp_path))

        coverages = []
        with tarfile.open(fileobj=sp_bytes, mode='r:gz') as tar:
            tiff_members = sorted(
                [m for m in tar.getmembers() if m.name.endswith('.tiff')],
                key=lambda m: m.name
            )[:N_TIFF_SAMPLE]

            for m in tiff_members:
                f = tar.extractfile(m)
                sp_data = tifffile.imread(io.BytesIO(f.read()))
                z = sp_data[..., 2] if sp_data.ndim == 3 else sp_data
                valid = (z > 0) & np.isfinite(z)
                coverages.append(float(valid.mean()) * 100)

        cov_arr = np.array(coverages)
        coverage_summary.append({
            'Keyframe':          kf.replace('keyframe_', 'KF'),
            'GT_frame0_%':       round(cov_arr[0], 1),
            'GT_median_%':       round(float(np.median(cov_arr)), 1),
            'GT_frame_last_%':   round(cov_arr[-1], 1),
            'Frames_utiles_%':   round(float((cov_arr > 30).mean() * 100), 1),
        })

        ax.plot(range(len(cov_arr)), cov_arr, color='seagreen', lw=2)
        ax.fill_between(range(len(cov_arr)), 30, cov_arr,
                         where=(cov_arr >= 30), alpha=0.2, color='green',
                         label='GT útil (>30%)')
        ax.fill_between(range(len(cov_arr)), 0, cov_arr,
                         where=(cov_arr < 30), alpha=0.2, color='red',
                         label='GT insuficiente (<30%)')
        ax.axhline(30, color='red',    ls='--', lw=1.2)
        ax.axhline(60, color='orange', ls='--', lw=1, label='buena cobertura (60%)')
        ax.set_title(f"{kf.replace('keyframe_','KF ')}\nmediana={np.median(cov_arr):.1f}%",
                     fontsize=10)
        ax.set_xlabel(f'Frame # (primeros {N_TIFF_SAMPLE})', fontsize=8)
        ax.set_ylabel('Cobertura GT (%)', fontsize=8)
        ax.set_ylim(0, 100)
        ax.grid(True, alpha=0.2)
        if ax == axes[0]: ax.legend(fontsize=7)

plt.suptitle(f'Cobertura GT en frames de video (primeros {N_TIFF_SAMPLE} frames)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / '26_cobertura_gt_video.png', dpi=150, bbox_inches='tight')
plt.show()

df_cov = pd.DataFrame(coverage_summary)
print(df_cov.to_string(index=False))
df_cov.to_csv(OUT_DIR / '26_cobertura_gt_video.csv', index=False)

### Cómo leer estas gráficas

El eje X son los primeros 30 frames del video. El eje Y es qué fracción de píxeles tiene GT válido.

**La zona verde** es donde la cobertura supera el 30% — umbral mínimo para supervisión útil. **La zona roja** es donde el GT es demasiado escaso.

**Patrones a observar:**
- **Curva que cae monotónicamente** → conforme el endoscopio se aleja del keyframe, el GT warpado cubre cada vez menos del frame. Esto es normal y esperado.
- **Caída brusca en los primeros frames** → el endoscopio se movió muy rápido al inicio. Los primeros frames tienen muy poco GT útil.
- **Curva estable** → el endoscopio se movió lentamente y el GT sigue siendo bueno en todos los frames del sample.

**`Frames_utiles_%`:** qué fracción de los primeros 30 frames tiene GT > 30%. Si es < 50%, solo la primera mitad del video es útil para supervisión con GT.

> **Nota importante:** el `scene_points.tar.gz` tiene ~197 frames en total, pero solo analizamos los primeros 30 para no leer 2.6 GB completos. La tendencia en los primeros 30 es indicativa del resto.

---
## Sección 7 — Tabla de aptitud por keyframe y modelo

### ¿Qué es la tabla de aptitud?

Consolidamos todos los indicadores de las secciones anteriores en una sola tabla que responde: **¿este keyframe es apto para entrenar/evaluar este modelo?**

Los criterios son:

| Indicador | Umbral OK | Umbral problemático |
|---|---|---|
| Frames borrosos | < 15% | > 30% |
| Flujo óptico mediano | 2–15 px | < 2 o > 20 px |
| Distancia total de trayectoria | > 20 mm | < 10 mm |
| Saltos de pose grandes | < 5% | > 15% |
| Cobertura GT mediana en video | > 50% | < 30% |

Cada criterio da un semáforo: verde (OK), amarillo (cuidado), rojo (problema).

In [ ]:
def semaforo(val, ok_thresh, bad_thresh, higher_is_better=True):
    if higher_is_better:
        if val >= ok_thresh:  return '✅'
        if val <= bad_thresh: return '❌'
        return '⚠️'
    else:
        if val <= ok_thresh:  return '✅'
        if val >= bad_thresh: return '❌'
        return '⚠️'

# Unir todos los dataframes por keyframe
kf_keys = [kf.replace('keyframe_', 'KF') for kf in KF_WITH_VIDEO]

rows = []
for kf_k in kf_keys:
    q = df_quality[df_quality['Keyframe'] == kf_k].iloc[0]  if len(df_quality) > 0  else None
    f = df_flow[df_flow['Keyframe']        == kf_k].iloc[0]  if len(df_flow) > 0    else None
    p = df_poses[df_poses['Keyframe']      == kf_k].iloc[0]  if len(df_poses) > 0   else None
    c = df_cov[df_cov['Keyframe']          == kf_k].iloc[0]  if len(df_cov) > 0     else None

    blur_pct    = q['Frames_borrosos_%']    if q is not None else float('nan')
    flow_med    = f['Flow_median_px']       if f is not None else float('nan')
    dist_mm     = p['Dist_total_mm']        if p is not None else float('nan')
    saltos_pct  = p['Saltos_grandes_%']     if p is not None else float('nan')
    gt_med      = c['GT_median_%']          if c is not None else float('nan')

    rows.append({
        'Keyframe':              kf_k,
        'Blur % (↓ mejor)':      f"{blur_pct:.1f}%  {semaforo(blur_pct, 15, 30, higher_is_better=False)}",
        'Flujo med. (px)':       f"{flow_med:.1f}  {'✅' if 2<=flow_med<=15 else ('⚠️' if flow_med<20 else '❌')}",
        'Trayect. (mm)':         f"{dist_mm:.0f}  {semaforo(dist_mm, 20, 10)}",
        'Saltos pose % (↓ mejor)':f"{saltos_pct:.1f}%  {semaforo(saltos_pct, 5, 15, higher_is_better=False)}",
        'GT video med. %':       f"{gt_med:.1f}%  {semaforo(gt_med, 50, 30)}",
    })

df_apt = pd.DataFrame(rows)
print("\n" + "="*85)
print("TABLA DE APTITUD POR KEYFRAME")
print("="*85)
print(df_apt.to_string(index=False))
print("="*85)
print("\n✅ = OK   ⚠️ = cuidado   ❌ = problemático")
print()
print("Aptitud por modelo:")
print("  NeXF / EndoGaussian    → requieren blur bajo + trayectoria > 20mm + flujo 2-15px")
print("  EndoDepthAndMotion     → requiere además saltos de pose < 5% y GT video > 50%")
df_apt.to_csv(OUT_DIR / '27_tabla_aptitud.csv', index=False)

### Cómo leer la tabla

Cada fila es un keyframe. Las columnas son los indicadores clave. El semáforo en cada celda es la evaluación rápida.

**Para elegir con qué keyframe empezar los experimentos:**
- El keyframe con **más ✅** es el más apto para todos los modelos.
- El keyframe con **más ❌** es el más difícil — útil para probar robustez, pero no para el primer experimento.

**Nota sobre los modelos:**
- **EndoDepthAndMotion** es el más exigente con la calidad de poses y GT del video. Un keyframe con saltos de pose grandes o GT < 30% puede hacer que el SLAM falle.
- **NeXF y EndoGaussian** son más tolerantes a poses ligeramente ruidosas, pero muy sensibles al blur (frames borrosos producen floaters en la reconstrucción).

---
## Resumen ejecutivo

### Lo que analizamos

| Análisis | Herramienta | Lo que nos dice |
|---|---|---|
| Metadata del video | OpenCV `VideoCapture` | Duración, framerate, resolución |
| Calidad de frames | Varianza del Laplaciano | Qué fracción de frames está borrosa |
| Movimiento entre frames | Flujo óptico Farneback | Si el movimiento es suficiente sin ser excesivo |
| Trayectoria de cámara | Poses R/T de JSONs | Si el endoscopio recorrió suficiente espacio |
| GT en frames de video | `scene_points.tar.gz` | Hasta qué frame el depth warpeado es útil |

### Implicaciones para el pipeline de experimentos

1. **Selección de keyframe:** usar el keyframe con mejor tabla de aptitud para los primeros experimentos con cada modelo.
2. **Selección de frames del video:** filtrar frames con Laplacian Variance < 50 y flujo > 20 px antes de pasar el video a los modelos.
3. **Submuestreo temporal:** si el flujo mediano es < 2 px/frame, submuestrear a 1 frame cada 5 para reducir redundancia.
4. **Umbral de GT:** para EndoDepthAndMotion, usar solo los primeros N frames donde la cobertura GT > 30%.
5. **Corrección de iluminación primero:** aplicar el método de corrección (del notebook 01) antes de pasar los frames a cualquier modelo, y medir si mejora el flujo óptico y la nitidez aparente.

### Siguiente paso

Con el EDA completo (notebooks 00, 01, 03) ya tenemos suficiente información para:
- Implementar el pipeline de corrección de iluminación.
- Configurar y correr NeXF, EndoGaussian y EndoDepthAndMotion sobre el keyframe elegido.
- Medir PSNR, SSIM, depth error y Chamfer Distance antes y después de la corrección.